# Countix Exercise Subset — Filter, Download & Validate

**Purpose:** Augment RepCount Part-A with exercise-matched clips from Countix (Kinetics-700).

**Pipeline:**
```
S3 annotation bundle (few MB, no AWS credentials needed)
        ↓  stream + SQL-style filter in memory
Filtered exercise rows (~600–900 clips)
        ↓  yt-dlp selective clip download
CV_Image_pose_detection/Data/Countix/video/{split}/{exercise}/{clip_id}.mp4
        ↓  RepCount-compatible annotation CSV
countix_repcount_format.csv  (same schema as train_cleaned.csv)
```

**Label mapping (Countix → your RepCount schema):**

| Countix class_label | Your type |
|---|---|
| squats | squat |
| pull-ups / pull ups | pull_up |
| push-ups / push ups | push_up |
| sit-ups / sit ups | sit_up |
| bench press | bench_pressing |
| front raises | front_raise |
| jumping jacks | jump_jacks |
| battle ropes | battle_rope |

**Resume-safe:** `download_log.csv` is written after every single clip.  
If Colab times out, re-run Cell 4 — completed clips are skipped automatically.

---
## Cell 1 — Mount Drive + Install dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install yt-dlp -q
!yt-dlp --version

import os

# ── CONFIG ── keep Countix inside the project tree ───────────────────────────
PROJECT_ROOT = '/content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection'
COUNTIX_ROOT = f'{PROJECT_ROOT}/Data/Countix'
VIDEO_DIR    = f'{COUNTIX_ROOT}/video'
ANNO_DIR     = f'{COUNTIX_ROOT}/annotation_cleaned'
LOG_PATH     = f'{ANNO_DIR}/download_log.csv'
FILTERED_CSV = f'{ANNO_DIR}/countix_filtered.csv'
REPCOUNT_CSV = f'{ANNO_DIR}/countix_repcount_format.csv'
ANNO_URL     = 'https://s3.amazonaws.com/kinetics/700_2020/annotations/countix.tar.gz'
SPOT_N       = 10   # clips to verify with OpenCV in Cell 6
# ─────────────────────────────────────────────────────────────────────────────

os.makedirs(VIDEO_DIR, exist_ok=True)
os.makedirs(ANNO_DIR,  exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')
print(f'Countix root: {COUNTIX_ROOT}')
print(f'Video output: {VIDEO_DIR}')
print(f'Log file    : {LOG_PATH}')

---
## Cell 2 — Stream S3 annotations → filter in memory

The full Countix tar.gz (~few MB) is downloaded once into memory.  
Only rows whose `class_label` maps into your exercise set are kept.  
The unfiltered CSV is **never written to disk**.

In [ ]:
import csv, io, re, tarfile, urllib.request
import pandas as pd

LABEL_MAP = {
    'squats': 'squat',
    'squat': 'squat',
    'pull ups': 'pull_up',
    'pullups': 'pull_up',
    'pull up': 'pull_up',
    'push ups': 'push_up',
    'pushups': 'push_up',
    'push up': 'push_up',
    'sit ups': 'sit_up',
    'situps': 'sit_up',
    'sit up': 'sit_up',
    'bench press': 'bench_pressing',
    'bench pressing': 'bench_pressing',
    'front raise': 'front_raise',
    'front raises': 'front_raise',
    'jumping jacks': 'jump_jacks',
    'jumping jack': 'jump_jacks',
    'jump jacks': 'jump_jacks',
    'jump jack': 'jump_jacks',
    'battle ropes': 'battle_rope',
    'battle rope': 'battle_rope',
    'pommel horse': 'pommelhorse',
    'pommelhorse': 'pommelhorse',
}

def normalize_label_key(value):
    text = value.strip().lower()
    text = re.sub(r'[_/-]+', ' ', text)
    text = re.sub(r'[^a-z0-9 ]+', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def map_countix_label(value):
    raw_label = value.strip().lower()
    mapped_label = LABEL_MAP.get(normalize_label_key(raw_label))
    return raw_label, mapped_label

def resolve_label_value(row):
    for key in ('class_label', 'label', 'class', 'action', 'category'):
        value = row.get(key)
        if value is not None and str(value).strip():
            return str(value)
    return ''

def resolve_video_id(row):
    for key in ('youtube_id', 'video_id', 'id'):
        value = row.get(key)
        if value is not None and str(value).strip():
            return str(value).strip()
    raise KeyError('Countix row is missing a video identifier column.')

def resolve_time_range(row):
    for start_key, end_key in (
        ('repetition_start', 'repetition_end'),
        ('time_start', 'time_end'),
        ('kinetics_start', 'kinetics_end'),
    ):
        start_value = row.get(start_key)
        end_value = row.get(end_key)
        if start_value is None or end_value is None:
            continue
        if not str(start_value).strip() or not str(end_value).strip():
            continue
        return float(start_value), float(end_value)
    raise KeyError('Countix row is missing a supported start/end time pair.')

def format_time_token(value):
    return f'{value:.3f}'.rstrip('0').rstrip('.').replace('.', 'p')

def build_clip_id(youtube_id, time_start, time_end):
    return f'{youtube_id}_{format_time_token(time_start)}_{format_time_token(time_end)}'

# ── Skip if already done ──────────────────────────────────────────────────────
if os.path.exists(FILTERED_CSV):
    print(f'Filtered CSV already exists — loading from {FILTERED_CSV}')
    try:
        df_filtered = pd.read_csv(FILTERED_CSV)
    except pd.errors.EmptyDataError:
        print('Filtered CSV is empty; regenerating from Countix annotations.')
        df_filtered = None
    else:
        if 'clip_id' not in df_filtered.columns:
            print('Filtered CSV uses the old youtube_id-only format; regenerating with clip-level IDs.')
            df_filtered = None
        elif len(df_filtered) == 0:
            print('Filtered CSV has zero rows; regenerating from Countix annotations.')
            df_filtered = None
else:
    df_filtered = None

if df_filtered is None:
    print('Streaming annotations from S3 ...')
    with urllib.request.urlopen(ANNO_URL) as r:
        raw = r.read()
    print(f'Bundle size: {len(raw)/1024:.1f} KB')

    rows = []
    dropped_labels = {}
    with tarfile.open(fileobj=io.BytesIO(raw), mode='r:gz') as tar:
        for m in tar.getmembers():
            if not m.name.endswith('.csv'):
                continue
            split = m.name.split('/')[-1].replace('countix_','').replace('.csv','')
            f = tar.extractfile(m)
            for row in csv.DictReader(io.TextIOWrapper(f, encoding='utf-8')):
                raw_label, mapped_label = map_countix_label(resolve_label_value(row))
                if mapped_label is None:
                    dropped_labels[raw_label] = dropped_labels.get(raw_label, 0) + 1
                    continue
                # handle both 'repetitions' and 'count' column names across versions
                rep = row.get('repetitions') or row.get('count') or '0'
                youtube_id = resolve_video_id(row)
                time_start, time_end = resolve_time_range(row)
                rows.append({
                    'clip_id':     build_clip_id(youtube_id, time_start, time_end),
                    'youtube_id':  youtube_id,
                    'time_start':  time_start,
                    'time_end':    time_end,
                    'count':       int(float(rep)),
                    'class_label': raw_label,
                    'type':        mapped_label,
                    'split':       split,
                })

    df_filtered = pd.DataFrame(rows, columns=[
        'clip_id', 'youtube_id', 'time_start', 'time_end',
        'count', 'class_label', 'type', 'split'
    ])
    df_filtered.to_csv(FILTERED_CSV, index=False)
    print(f'Saved → {FILTERED_CSV}')

print(f'\nTotal clips after filter: {len(df_filtered)}')
if len(df_filtered):
    print('\nPer-exercise / per-split breakdown:')
    print(df_filtered.groupby(['type','split']).size().unstack(fill_value=0).to_string())
    print('\nRepetition count distribution:')
    print(df_filtered['count'].describe().round(2))
else:
    print('\nNo rows matched the current Countix label filter.')
if 'dropped_labels' in globals() and dropped_labels:
    print('\nDropped out-of-scope / unmapped labels:')
    for label, count in sorted(dropped_labels.items(), key=lambda item: (-item[1], item[0]))[:25]:
        print(f'  {label or "<empty>"}: {count}')

---
## Cell 3 — (Optional) Inspect annotation columns

Run this if Cell 2 threw a KeyError — it prints the raw column names  
so you can adjust the `rep` fallback or `class_label` field name.

In [ ]:
import urllib.request, tarfile, io, csv

print('Inspecting annotation columns ...')
with urllib.request.urlopen(ANNO_URL) as r:
    raw = r.read()

with tarfile.open(fileobj=io.BytesIO(raw), mode='r:gz') as tar:
    for m in tar.getmembers():
        if m.name.endswith('.csv'):
            f = tar.extractfile(m)
            reader = csv.DictReader(io.TextIOWrapper(f, encoding='utf-8'))
            print(f'\n{m.name}')
            print(f'  Columns : {reader.fieldnames}')
            rows = [next(reader) for _ in range(3)]
            for row in rows:
                print(f'  Sample  : {dict(row)}')
            break

---
## Cell 4 — Download clips  *(resume-safe)*

- Each clip is saved to `VIDEO_DIR/{split}/{type}/{clip_id}.mp4`
- `download_log.csv` is updated after **every** clip
- Re-run this cell after a Colab timeout — done clips are skipped
- Expected 10–20% unavailability from deleted/private YouTube videos

In [ ]:
import subprocess, time
import pandas as pd
from pathlib import Path

df_filtered = pd.read_csv(FILTERED_CSV)

# ── Load resume log ───────────────────────────────────────────────────────────
if os.path.exists(LOG_PATH):
    df_log   = pd.read_csv(LOG_PATH)
    if 'clip_id' not in df_log.columns:
        print('Existing download log uses the old youtube_id-only format; starting a new clip-level log.')
        df_log = pd.DataFrame(columns=['clip_id','youtube_id','type','split','status','path'])
        done_ids = set()
    else:
        done_ids = set(df_log['clip_id'].tolist())
        print(f'Resuming — {len(done_ids)} clips already processed.')
else:
    df_log   = pd.DataFrame(columns=['clip_id','youtube_id','type','split','status','path'])
    done_ids = set()
    print('Starting fresh download.')
    df_log.to_csv(LOG_PATH, index=False)

pending = df_filtered[~df_filtered['clip_id'].isin(done_ids)]
print(f'Clips pending this session: {len(pending)} / {len(df_filtered)} total\n')

new_rows = []
for i, row in enumerate(pending.itertuples(), 1):
    out_dir  = Path(VIDEO_DIR) / row.split / row.type
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f'{row.clip_id}.mp4'

    cmd = [
        'yt-dlp',
        f'https://www.youtube.com/watch?v={row.youtube_id}',
        '--download-sections', f'*{row.time_start}-{row.time_end}',
        '--format', 'mp4/bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]',
        '-o', str(out_path),
        '--quiet', '--no-warnings', '--no-playlist',
    ]

    try:
        res    = subprocess.run(cmd, capture_output=True, timeout=90)
        status = 'ok' if (res.returncode == 0 and out_path.exists()) else 'failed'
    except subprocess.TimeoutExpired:
        status = 'timeout'

    entry = {
        'clip_id': row.clip_id,
        'youtube_id': row.youtube_id, 'type': row.type,
        'split': row.split, 'status': status,
        'path': str(out_path) if status == 'ok' else '',
    }
    new_rows.append(entry)
    df_log = pd.concat([df_log, pd.DataFrame([entry])], ignore_index=True)
    df_log.to_csv(LOG_PATH, index=False)   # checkpoint after every clip

    if i % 10 == 0 or i == len(pending):
        ok_n   = sum(r['status'] == 'ok'  for r in new_rows)
        fail_n = sum(r['status'] != 'ok'  for r in new_rows)
        print(f'  [{i:>4}/{len(pending)}]  ok={ok_n}  failed/timeout={fail_n}')

    time.sleep(0.3)

print('\nSession complete.')

---
## Cell 5 — Download summary report

In [ ]:
import pandas as pd, os

df_log      = pd.read_csv(LOG_PATH) if os.path.exists(LOG_PATH) else pd.DataFrame(columns=['clip_id','youtube_id','type','split','status','path'])
df_filtered = pd.read_csv(FILTERED_CSV)
ok     = df_log[df_log['status'] == 'ok']
failed = df_log[df_log['status'] != 'ok']

print('=' * 52)
print('DOWNLOAD SUMMARY')
print('=' * 52)
print(f'Total clips targeted      : {len(df_filtered)}')
print(f'Successfully downloaded   : {len(ok)}')
print(f'Failed / unavailable      : {len(failed)}')
print(f'Availability rate         : {len(ok)/max(len(df_filtered),1)*100:.1f}%')

if len(ok):
    print('\n--- OK clips per exercise / split ---')
    print(ok.groupby(['type','split']).size().unstack(fill_value=0).to_string())

if len(failed):
    print('\n--- Failed clips per exercise ---')
    print(failed['type'].value_counts().to_string())
else:
    print('\nNo failures.')

total_bytes = sum(
    os.path.getsize(p) for p in ok['path'].dropna() if os.path.exists(p)
)
print(f'\nTotal disk usage: {total_bytes/1e9:.2f} GB')

---
## Cell 6 — Build RepCount-compatible annotation CSV

Produces `countix_repcount_format.csv` with the same schema as  
`train_cleaned.csv` / `valid_cleaned.csv`:  
`name | type | count | split | time_start | time_end | source`

In [ ]:
import pandas as pd

df_log      = pd.read_csv(LOG_PATH) if os.path.exists(LOG_PATH) else pd.DataFrame(columns=['clip_id','youtube_id','type','split','status','path'])
df_filtered = pd.read_csv(FILTERED_CSV)

ok_ids = set(df_log[df_log['status'] == 'ok']['clip_id'])
df_ok  = df_filtered[df_filtered['clip_id'].isin(ok_ids)].copy()

df_repcount = pd.DataFrame({
    'name':       df_ok['clip_id'] + '.mp4',
    'type':       df_ok['type'],
    'count':      df_ok['count'],
    'split':      df_ok['split'],
    'time_start': df_ok['time_start'],
    'time_end':   df_ok['time_end'],
    'source':     'countix',
})

df_repcount.to_csv(REPCOUNT_CSV, index=False)
print(f'Saved → {REPCOUNT_CSV}')
print(f'Rows  : {len(df_repcount)}')
if len(df_repcount):
    print('\nPer-type / per-split:')
    print(df_repcount.groupby(['type','split']).size().unstack(fill_value=0).to_string())
    print('\nSample rows:')
    df_repcount.head(5)
else:
    print('\nNo filtered Countix clips were downloaded, so the RepCount-format CSV is empty.')

---
## Cell 7 — OpenCV spot-check

Verifies a random sample of downloaded clips are readable and non-corrupt  
before you run YOLO pose extraction on the full set.

In [ ]:
import cv2, os, random
import pandas as pd

if not os.path.exists(LOG_PATH):
    print('No download log found yet. Run the download cell first.')
else:
    df_log   = pd.read_csv(LOG_PATH)
    ok_paths = df_log[df_log['status'] == 'ok']['path'].dropna().tolist()
    if not ok_paths:
        print('No successfully downloaded clips available for spot-checking.')
    else:
        sample   = random.sample(ok_paths, min(SPOT_N, len(ok_paths)))
        print(f'Spot-checking {len(sample)} clips ...\n')
        results = []
        for path in sample:
            cap      = cv2.VideoCapture(path)
            readable = cap.isOpened()
            fps      = cap.get(cv2.CAP_PROP_FPS)              if readable else 0
            nf       = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) if readable else 0
            w        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))  if readable else 0
            h        = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)) if readable else 0
            cap.release()
            status = 'OK' if (readable and nf > 0) else 'CORRUPT'
            results.append(status)
            print(f'  {status:7s}  {path.split("/")[-1]:35s}  {w}x{h}  {fps:.1f}fps  {nf} frames')
        print(f'\n{results.count("OK")}/{len(results)} clips readable.')